In [1]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Patient Table

In [4]:
# Load the data
patient_query = """
SELECT  pat.subject_id,
        pat.anchor_age

FROM    physionet-data.mimiciv_3_1_hosp.patients pat
"""
# patient_query_job = client.query(patient_query)
# patient_df = patient_query_job.result().to_dataframe()
df_age = client.query(patient_query).to_dataframe()

print(df_age.info())
print(df_age.describe())
print("Nulls:\n", df_age.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   subject_id  364627 non-null  Int64
 1   anchor_age  364627 non-null  Int64
dtypes: Int64(2)
memory usage: 6.3 MB
None
            subject_id  anchor_age
count         364627.0    364627.0
mean   15011664.762544   48.875097
std     2885012.874923   20.943316
min         10000032.0        18.0
25%         12512008.5        29.0
50%         15018682.0        48.0
75%         17509003.0        65.0
max         19999987.0        91.0
Nulls:
 subject_id    0
anchor_age    0
dtype: int64


In [5]:
# Handle the 91+ anonymization artifact
## MIMIC-IV shifts ages >89 to 91 to protect patient identity.
## Two approaches:
### 1) Treat 91 as a sentinel ">=90" value (keep it, acknowledge in docs)
### 2) Clip everything at 89 to remove ambiguity
## Go with Option 1

In [30]:
# Bin into clinical age groups
bins = [0, 18, 40, 65, 80, 200]
labels = ["<18", "18-40", "41-65", "66-80", "81+"]

df_age["age_group"] = pd.cut(
    df_age["anchor_age"],
    bins=bins,
    labels=labels,
    right=True
)

# # One-hot encode if your model needs it (tree-based models often prefer ordinal)
# age_dummies = pd.get_dummies(df_age["age_group"], prefix="age_grp")
# df_age = pd.concat([df_age, age_dummies], axis=1)

In [7]:
# Normalize continuous age
scaler = MinMaxScaler()
df_age["anchor_age_norm"] = scaler.fit_transform(df_age[["anchor_age"]])

In [8]:
# Final cleaned table
df_age_final = df_age[[
    "subject_id",
    "anchor_age",           # original continuous feature (91 = sentinel for ≥90)
    "anchor_age_norm",      # min-max scaled for RL input
    "age_group",            # categorical label
    *[c for c in df_age.columns if c.startswith("age_grp_")]
]]

print(df_age_final.head())
print(df_age_final.shape)

   subject_id  anchor_age  anchor_age_norm age_group
0    10078138          18              0.0       <18
1    10180372          18              0.0       <18
2    10686175          18              0.0       <18
3    10851602          18              0.0       <18
4    10902424          18              0.0       <18
(364627, 4)


# Data Preparation - ICU from ED Transfers

In [9]:
# Load the data
icu_from_ed_query = """
WITH ed_admissions AS (
    -- identify hospital admissions that came through the ED
    SELECT
        hadm_id,
        subject_id,
        admittime,
        admission_type,
        admission_location
    FROM physionet-data.mimiciv_3_1_hosp.admissions
    WHERE admission_location IN (
        'EMERGENCY ROOM',
        'TRANSFER FROM EMERGENCY ROOM'
    )
),

icu_stays AS (
    -- get first ICU stay per hospital admission
    SELECT
        subject_id,
        hadm_id,
        stay_id,
        first_careunit,
        intime  AS icu_intime,
        outtime AS icu_outtime,
        los     AS icu_los_days,
        ROW_NUMBER() OVER (
            PARTITION BY hadm_id
            ORDER BY intime ASC
        ) AS stay_rank
    FROM physionet-data.mimiciv_3_1_icu.icustays
),

ed_transfers AS (
    -- confirm the transfer pathway went through the ED
    SELECT
        t.hadm_id,
        t.subject_id,
        MIN(t.intime)  AS ed_intime,
        MAX(t.outtime) AS ed_outtime
    FROM physionet-data.mimiciv_3_1_hosp.transfers t
    WHERE t.careunit IN (
        'Emergency Department',
        'Emergency Department Observation'
    )
    GROUP BY t.hadm_id, t.subject_id
),

clinician_verified AS (
    -- confirm a clinician actually charted on this patient
    -- during their ICU stay window
    -- at least 2 chart events required to avoid
    -- single accidental entries
    SELECT
        ce.subject_id,
        ce.stay_id,
        COUNT(*)                    AS n_chart_events,
        COUNT(DISTINCT ce.caregiver_id) AS n_unique_caregivers,
        MIN(ce.charttime)           AS first_chart_time,
        MAX(ce.charttime)           AS last_chart_time
    FROM physionet-data.mimiciv_3_1_icu.chartevents ce
    INNER JOIN physionet-data.mimiciv_3_1_icu.icustays i
        ON ce.stay_id = i.stay_id
    WHERE ce.charttime BETWEEN i.intime AND i.outtime
    GROUP BY ce.subject_id, ce.stay_id
    HAVING COUNT(*) >= 2                -- at least 2 charted events
       AND COUNT(DISTINCT ce.caregiver_id) >= 1  -- by at least 1 caregiver
)

SELECT
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    i.first_careunit,
    i.icu_intime,
    i.icu_outtime,
    i.icu_los_days,
    et.ed_intime,
    et.ed_outtime,
    TIMESTAMP_DIFF(i.icu_intime, et.ed_outtime, HOUR) AS hrs_ed_to_icu,
    ea.admission_type,
    ea.admission_location,
    cv.n_chart_events,
    cv.n_unique_caregivers,
    cv.first_chart_time,
    cv.last_chart_time
FROM icu_stays i
INNER JOIN ed_admissions ea
    ON i.hadm_id = ea.hadm_id
INNER JOIN ed_transfers et
    ON i.hadm_id = et.hadm_id
INNER JOIN clinician_verified cv           -- INNER JOIN drops patients
    ON i.stay_id = cv.stay_id             -- with no clinical contact
WHERE i.stay_rank = 1                      -- first ICU stay only
  AND i.icu_intime > et.ed_intime          -- ICU came after ED
  AND TIMESTAMP_DIFF(
        i.icu_intime, et.ed_outtime, HOUR
      ) <= 24                              -- within 24 hours of leaving ED
ORDER BY i.subject_id
"""

df_icu = client.query(icu_from_ed_query).to_dataframe()

print(df_icu.shape)
print(df_icu.dtypes)
print(df_icu.isnull().sum())

(29733, 16)
subject_id                      Int64
hadm_id                         Int64
stay_id                         Int64
first_careunit                 object
icu_intime             datetime64[us]
icu_outtime            datetime64[us]
icu_los_days                  float64
ed_intime              datetime64[us]
ed_outtime             datetime64[us]
hrs_ed_to_icu                   Int64
admission_type                 object
admission_location             object
n_chart_events                  Int64
n_unique_caregivers             Int64
first_chart_time       datetime64[us]
last_chart_time        datetime64[us]
dtype: object
subject_id             0
hadm_id                0
stay_id                0
first_careunit         0
icu_intime             0
icu_outtime            0
icu_los_days           0
ed_intime              0
ed_outtime             0
hrs_ed_to_icu          0
admission_type         0
admission_location     0
n_chart_events         0
n_unique_caregivers    0
first_chart_time

In [10]:
# Cast and parse timestamps
ts_cols = ["icu_intime", "icu_outtime", "ed_intime", "ed_outtime",
           "first_chart_time", "last_chart_time"]

for col in ts_cols:
    df_icu[col] = pd.to_datetime(df_icu[col], utc=True)

"""
Temporal context features derived from ED arrival
Hour-of-day and day-of-week capture shift patterns, staffing levels,
and bed availability — all clinically meaningful for transfer lag
"""
df_icu["ed_arrival_hour"]    = df_icu["ed_intime"].dt.hour
df_icu["ed_arrival_dow"]     = df_icu["ed_intime"].dt.dayofweek   # 0=Mon, 6=Sun
df_icu["ed_arrival_weekend"] = (df_icu["ed_arrival_dow"] >= 5).astype(int)

In [11]:
# Validate and clean durations
# hrs_ed_to_icu: drop negative values — these represent broken transfer pathway
# reconstructions (median -70 hrs, range -601 to -1), not documentation lag.
# At 0.5% of the dataset the loss is negligible.
print(f"Rows before: {len(df_icu)}")
neg_mask = df_icu["hrs_ed_to_icu"] < 0
df_icu = df_icu[~neg_mask].copy()
print(f"Rows after:  {len(df_icu)}")
print(f"Dropped:     {neg_mask.sum()} rows")

assert (df_icu["hrs_ed_to_icu"] >= 0).all()
assert (df_icu["hrs_ed_to_icu"] <= 24).all(), "Transfer times exceed 24h cap"

# icu_los_days: zero/fractional values are clinically valid (rapid deterioration,
# early death, or fast transfer) — retain them, just confirm no negatives
neg_los = df_icu[df_icu["icu_los_days"] < 0]
if not neg_los.empty:
    print(f"Warning: {len(neg_los)} rows with negative icu_los_days — review these")
    df_icu = df_icu[df_icu["icu_los_days"] >= 0].copy()

Rows before: 29733
Rows after:  29585
Dropped:     148 rows


In [12]:
# Encode categoricals
# first_careunit: nominal — which ICU type the patient landed in
careunit_dummies = pd.get_dummies(df_icu["first_careunit"], prefix="careunit")
df_icu = pd.concat([df_icu, careunit_dummies], axis=1)

# admission_type: nominal categories in MIMIC (URGENT, ELECTIVE, etc.)
admtype_dummies = pd.get_dummies(df_icu["admission_type"], prefix="adm_type")
df_icu = pd.concat([df_icu, admtype_dummies], axis=1)

In [13]:
# Derive acuity signal features
# Chart rate: events per hour of ICU stay — captures care intensity
df_icu["icu_los_hrs"] = df_icu["icu_los_days"] * 24
df_icu["icu_los_hrs"] = df_icu["icu_los_hrs"].replace(0, 0.5)   # avoid div/0 for very short stays

df_icu["chart_rate_per_hr"] = df_icu["n_chart_events"] / df_icu["icu_los_hrs"]

# Charting span: hours between first and last chart event
# Wide span = sustained monitoring; narrow span = concentrated early activity
df_icu["chart_span_hrs"] = (
    (df_icu["last_chart_time"] - df_icu["first_chart_time"])
    .dt.total_seconds() / 3600
)

# Caregiver ratio: unique caregivers per chart event
# High ratio = many hands involved = higher acuity or complex case
df_icu["caregiver_ratio"] = df_icu["n_unique_caregivers"] / df_icu["n_chart_events"]

In [14]:
# Normalize continuous features
scaler = MinMaxScaler()

continuous_cols = [
    "hrs_ed_to_icu",
    "icu_los_days",
    "n_chart_events",
    "n_unique_caregivers",
    "chart_rate_per_hr",
    "chart_span_hrs",
    "caregiver_ratio",
    "ed_arrival_hour",
]

norm_col_names = [f"{c}_norm" for c in continuous_cols]
df_icu[norm_col_names] = scaler.fit_transform(df_icu[continuous_cols])

In [15]:
# Final table
# Retain all three join keys; drop raw timestamps before handoff
# admission_location retained for audit/stratification but carries no
# discriminating model signal — every row is ED by construction
id_cols      = ["subject_id", "hadm_id", "stay_id"]
raw_features = [
    "first_careunit", "admission_type", "admission_location",
    "hrs_ed_to_icu", "icu_los_days",
    "n_chart_events", "n_unique_caregivers",
    "chart_rate_per_hr", "chart_span_hrs", "caregiver_ratio",
    "ed_arrival_hour", "ed_arrival_dow", "ed_arrival_weekend",
]
derived_norm = norm_col_names
encoded_cats = (
    [c for c in df_icu.columns if c.startswith("careunit_")] +
    [c for c in df_icu.columns if c.startswith("adm_type_")]
)

df_icu_final = df_icu[id_cols + raw_features + derived_norm + encoded_cats].copy()

print(df_icu_final.shape)
print(df_icu_final.head())

(29585, 39)
   subject_id   hadm_id   stay_id  \
0    10000032  29079034  39553978   
1    10000690  25860671  37081114   
2    10000980  26913865  39765666   
3    10002155  28994087  31090461   
4    10002155  20345487  32358465   

                                     first_careunit admission_type  \
0                Medical Intensive Care Unit (MICU)       EW EMER.   
1                Medical Intensive Care Unit (MICU)       EW EMER.   
2                Medical Intensive Care Unit (MICU)       EW EMER.   
3  Medical/Surgical Intensive Care Unit (MICU/SICU)       EW EMER.   
4                Medical Intensive Care Unit (MICU)       EW EMER.   

  admission_location  hrs_ed_to_icu  icu_los_days  n_chart_events  \
0     EMERGENCY ROOM              0      0.410266             473   
1     EMERGENCY ROOM              0      3.893252            3836   
2     EMERGENCY ROOM              0      0.497535             446   
3     EMERGENCY ROOM              0      3.891447            3184   

# Data Preparation - Machine Measurements

In [16]:
# Load the data
machine_measurements_query = """
SELECT  mm.subject_id,
        mm.report_0,
        mm.report_1,
        mm.report_2,
        mm.report_3,
        mm.report_4,
        mm.report_5,
        mm.report_6,
        mm.report_7,
        mm.report_8,
        mm.report_9,
        mm.report_10,
        mm.report_11,
        mm.report_12,
        mm.report_13,
        mm.report_14,
        mm.report_15,
        mm.report_16,
        mm.report_17

FROM    physionet-data.mimiciv_ecg.machine_measurements mm
"""

machine_measurements_df = client.query(machine_measurements_query).to_dataframe()

print(machine_measurements_df.shape)
print(machine_measurements_df.dtypes)

(800035, 19)
subject_id     Int64
report_0      object
report_1      object
report_2      object
report_3      object
report_4      object
report_5      object
report_6      object
report_7      object
report_8      object
report_9      object
report_10     object
report_11     object
report_12     object
report_13     object
report_14     object
report_15     object
report_16     object
report_17     object
dtype: object


In [17]:
# Define regex patterns
# Each report_i column is an independent ECG machine finding.
# We scan each column separately to preserve finding count and avoid
# cross-column boundary artifacts from concatenation.

negative_patterns = re.compile(r"""
    ACUTE\s+INFARCT|
    INFARCT(?:ION)?|
    ISCH[EAE]MIA|
    MYOCARDIAL\s+INJURY|
    STRONGLY\s+SUGGESTS|
    CONSIDER\s+ACUTE|
    POSSIBLY\s+ACUTE|
    PROBABLY\s+ACUTE|
    COMPLETE\s+AV\s+BLOCK|
    LEFT\s+BUNDLE\s+BRANCH\s+BLOCK|
    RIGHT\s+BUNDLE\s+BRANCH\s+BLOCK|
    VENTRICULAR\s+TACHYCARDIA|
    ATRIAL\s+FIBRILLATION|
    ATRIAL\s+FLUTTER|
    VENTRICULAR\s+HYPERTROPHY|
    PROLONGED\s+QT|
    LONG\s+QTc|
    PERICARDITIS|
    ST\s+ELEVATION.*INFARCT|
    WIDE[\s\-]QRS\s+TACHYCARDIA|
    COMPLETE\s+HEART\s+BLOCK|
    EXTREME\s+TACHYCARDIA|
    VENTRICULAR\s+PREEXCITATION|
    WPW\s+PATTERN
""", re.IGNORECASE | re.VERBOSE)

neutral_patterns = re.compile(r"""
    BORDERLINE|
    POSSIBLE(?!\s+ACUTE)|
    PROBABLE(?!\s+ACUTE)|
    NONSPECIFIC|
    NON[\s\-]SPECIFIC|
    CANNOT\s+RULE\s+OUT|
    MAY\s+BE\s+DUE|
    EQUIVOCAL|
    ABNORMAL\s+FOR\s+AGE|
    CONSIDER(?!\s+ACUTE)|
    AGE\s+UNDETERMINED|
    AGE\s+INDETERMINATE|
    ST[\s\-]T\s+CHANGES|
    T\s+WAVE\s+CHANGES|
    REPOLARIZATION\s+ABNORMALITY|
    CONDUCTION\s+DEFECT|
    AXIS\s+DEVIATION|
    ATRIAL\s+ABNORMALITY|
    ATRIAL\s+ENLARGEMENT|
    LOW\s+QRS\s+VOLTAGE|
    POOR\s+R\s+WAVE\s+PROGRESSION|
    ALL\s+12\s+LEADS\s+ARE\s+MISSING|
    NOT\s+ENOUGH\s+LEADS|
    DATA\s+QUALITY|
    UNSUITABLE\s+FOR\s+ANALYSIS|
    RECORDING\s+UNSUITABLE|
    TECHNICAL\s+ERROR|
    ANALYSIS\s+ERROR|
    PEDIATRIC|
    MEASUREMENT\s+ERROR|
    NO\s+FURTHER\s+ANALYSIS
""", re.IGNORECASE | re.VERBOSE)

# Anchored to full cell value — strip() is called before matching,
# so ^ and $ reliably catch cells that are purely these phrases
normal_patterns = re.compile(r"""
    ^\s*NORMAL\s+ECG\s*$|
    WITHIN\s+NORMAL\s+LIMITS|
    NORMAL\s+VARIANT|
    AVAILABLE\s+LEADS\s+NORMAL|
    NORMAL\s+ECG\s+BASED\s+ON|
    NORMAL\s+ECG\s+EXCEPT\s+FOR\s+RATE|
    ^SINUS\s+RHYTHM\s*\.?\s*$
""", re.IGNORECASE | re.VERBOSE)

In [18]:
# Classify each cell
# Priority order: negative > neutral > normal > empty
# A cell with "POSSIBLE ACUTE INFARCT" matches both neutral (POSSIBLE) and
# negative (ACUTE INFARCT) — negative wins because it is checked first
def classify_cell(value):
    if value is None or str(value).strip() in ["nan", "", "None"]:
        return "empty"
    v = str(value).strip()
    if negative_patterns.search(v):
        return "negative"
    elif neutral_patterns.search(v):
        return "neutral"
    elif normal_patterns.search(v):
        return "normal"
    else:
        return "neutral"   # unrecognized findings treated as neutral, not normal

In [19]:
# Apply classification and ordinal encoding
report_cols = [f"report_{i}" for i in range(18)]
label_cols  = [f"report_{i}_label" for i in range(18)]
score_cols  = [f"report_{i}_label_score" for i in range(18)]

# Ordinal: empty=0, normal=1, neutral=2, negative=3
ordinal_map = {"empty": 0, "normal": 1, "neutral": 2, "negative": 3}

for report_col, label_col, score_col in zip(report_cols, label_cols, score_cols):
    machine_measurements_df[label_col] = (
        machine_measurements_df[report_col].apply(classify_cell)
    )
    machine_measurements_df[score_col] = (
        machine_measurements_df[label_col].map(ordinal_map)
    )

In [20]:
# Flag patients with no ECG conducted
# All 18 label columns empty = no ECG was performed or recorded
machine_measurements_df["no_ecg"] = (
    machine_measurements_df[label_cols]
    .eq("empty")
    .all(axis=1)
    .astype(int)
)

In [21]:
# Aggregate summary features
machine_measurements_df["ecg_negative_count"] = (
    machine_measurements_df[label_cols].eq("negative").sum(axis=1)
)
machine_measurements_df["ecg_neutral_count"] = (
    machine_measurements_df[label_cols].eq("neutral").sum(axis=1)
)
machine_measurements_df["ecg_normal_count"] = (
    machine_measurements_df[label_cols].eq("normal").sum(axis=1)
)
machine_measurements_df["ecg_max_severity"] = (
    machine_measurements_df[score_cols].max(axis=1)
)
# Mean severity excludes empty cells (score=0) so the average reflects
# only findings that were actually present
machine_measurements_df["ecg_mean_severity"] = (
    machine_measurements_df[score_cols]
    .replace(0, pd.NA)     # exclude empty (ordinal 0) from the mean
    .mean(axis=1)
    .fillna(0)             # patients with all-empty scores get 0
    .round(2)
)
machine_measurements_df["ecg_any_negative"] = (
    machine_measurements_df[label_cols].eq("negative").any(axis=1).astype(int)
)
machine_measurements_df["ecg_total_findings"] = (
    machine_measurements_df[label_cols].ne("empty").sum(axis=1)
)

/tmp/ipykernel_34852/163691471.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)             # patients with all-empty scores get 0


In [22]:
# Final table
ecg_features = [
    "subject_id",
    "ecg_any_negative",
    "ecg_max_severity",
    "ecg_negative_count",
    "ecg_neutral_count",
    "ecg_normal_count",
    "ecg_mean_severity",
    "ecg_total_findings",
    "no_ecg",
]

ecg_summary_df = machine_measurements_df[ecg_features].copy()

print(ecg_summary_df.describe())
print(f"\nPatients with no ECG conducted:              {ecg_summary_df['no_ecg'].sum():,}")
print(f"No ECG rate:                                 {ecg_summary_df['no_ecg'].mean():.1%}")
print(f"Patients with at least one negative finding: {ecg_summary_df['ecg_any_negative'].sum():,}")
print(f"Negative finding rate:                       {ecg_summary_df['ecg_any_negative'].mean():.1%}")

            subject_id  ecg_any_negative  ecg_max_severity  \
count         800035.0     800035.000000     800035.000000   
mean   15003193.384267          0.464022          2.305433   
std     2877969.929111          0.498704          0.727548   
min         10000032.0          0.000000          0.000000   
25%         12503687.0          0.000000          2.000000   
50%         15000407.0          0.000000          2.000000   
75%         17496985.0          1.000000          3.000000   
max         19999987.0          1.000000          3.000000   

       ecg_negative_count  ecg_neutral_count  ecg_normal_count  \
count       800035.000000      800035.000000     800035.000000   
mean             0.752360           2.012053          0.685583   
std              0.991838           1.391983          0.711073   
min              0.000000           0.000000          0.000000   
25%              0.000000           1.000000          0.000000   
50%              0.000000           2.000000 

# Data Preparation - Radiology

In [23]:
# Load the data
radiology_query = """
SELECT  rad.subject_id,
        rad.note_id,
        rad.charttime,
        rad.storetime,
        det.field_name,
        det.field_value

FROM    physionet-data.mimiciv_note.radiology rad

        INNER JOIN physionet-data.mimiciv_note.radiology_detail det
        ON rad.note_id = det.note_id
"""
df_rad = client.query(radiology_query).to_dataframe()

print(df_rad.shape)
print(df_rad["field_name"].value_counts())   # audit all field types before filtering

df_rad["charttime"] = pd.to_datetime(df_rad["charttime"], utc=True)
df_rad["storetime"] = pd.to_datetime(df_rad["storetime"], utc=True)

(6021832, 6)
field_name
exam_code           2902561
exam_name           2902561
cpt_code             165504
parent_note_id        25603
addendum_note_id      25603
Name: count, dtype: int64


In [24]:
# Filter to exam_name rows and clean
df_rad = df_rad[df_rad["field_name"] == "exam_name"].copy()
df_rad["exam_name"] = df_rad["field_value"].str.upper().str.strip()

# Drop administrative and billing rows — these are not imaging exams
admin_patterns = [
    "BY SAME PHYSICIAN", "BY DIFFERENT PHYSICIAN",
    "DISTINCT PROCEDURAL SERVICE", "SEPARATE STRUCTURE",
    "MOD SEDATION", "FEE ADJUSTED", "___", "CAD "
]
mask_admin = df_rad["exam_name"].str.contains("|".join(admin_patterns), na=False)
df_rad = df_rad[~mask_admin].copy()

print(f"\nRows after filtering to exam_name and removing admin: {len(df_rad)}")


Rows after filtering to exam_name and removing admin: 2623806


In [25]:
# Resolve field_value format
# field_value can be one of three formats:
#   a) CPT code       — purely numeric, e.g. "71046"
#   b) Exam code      — short uppercase label, e.g. "CHEST (PORTABLE AP)"
#   c) Literal name   — verbose, e.g. "COMPUTED TOMOGRAPHY CHEST W/CONTRAST"
#
# CPT codes fall through string matching silently, landing in 'Other'.
# The lookup below maps common radiology CPT codes to (modality, body_region)
# before string matching runs, preventing that silent failure.

cpt_lookup = {
    # Chest X-ray
    "71045": ("X-ray",      "Chest"),
    "71046": ("X-ray",      "Chest"),
    "71047": ("X-ray",      "Chest"),
    "71048": ("X-ray",      "Chest"),
    # CT Chest
    "71250": ("CT",         "Chest"),
    "71260": ("CT",         "Chest"),
    "71270": ("CT",         "Chest"),
    # CTA Chest
    "71275": ("CTA",        "Chest"),
    # CT Abdomen/Pelvis
    "74150": ("CT",         "Abdomen"),
    "74160": ("CT",         "Abdomen"),
    "74170": ("CT",         "Abdomen"),
    "74177": ("CT",         "Abdomen"),
    "74178": ("CT",         "Abdomen"),
    "74179": ("CT",         "Abdomen"),
    "72192": ("CT",         "Pelvis/GYN"),
    "72193": ("CT",         "Pelvis/GYN"),
    "72194": ("CT",         "Pelvis/GYN"),
    # CT Head/Brain
    "70450": ("CT",         "Head/Neuro"),
    "70460": ("CT",         "Head/Neuro"),
    "70470": ("CT",         "Head/Neuro"),
    # CTA Head/Neck
    "70496": ("CTA",        "Head/Neuro"),
    "70498": ("CTA",        "Head/Neuro"),
    # MRI Brain
    "70551": ("MRI",        "Head/Neuro"),
    "70552": ("MRI",        "Head/Neuro"),
    "70553": ("MRI",        "Head/Neuro"),
    # MRI Spine
    "72141": ("MRI",        "Spine"),
    "72146": ("MRI",        "Spine"),
    "72148": ("MRI",        "Spine"),
    # Ultrasound Abdomen
    "76700": ("Ultrasound", "Abdomen"),
    "76705": ("Ultrasound", "Abdomen"),
    # Ultrasound Pelvis
    "76856": ("Ultrasound", "Pelvis/GYN"),
    "76857": ("Ultrasound", "Pelvis/GYN"),
    # Echocardiogram
    "93306": ("Ultrasound", "Cardiac"),
    "93307": ("Ultrasound", "Cardiac"),
    "93308": ("Ultrasound", "Cardiac"),
    # Fluoroscopy / line placement
    "77001": ("Fluoroscopy", "Vascular"),
    "77002": ("Fluoroscopy", "Vascular"),
}

def is_cpt_code(value):
    return bool(re.match(r"^\d{5}$", str(value).strip()))

In [26]:
# Assign modality and body region
# Resolution order: CPT lookup → string matching → fallback 'Other'

def assign_modality(name):
    if is_cpt_code(name):
        return cpt_lookup.get(name.strip(), ("Other", "Other"))[0]
    if "CTA " in name:
        return "CTA"
    if any(x in name for x in ["CT ", " CT", "CT ABD", "CT HEAD",
                                "CT CHEST", "CT C-SPINE", "CT PELVIS",
                                "COMPUTED TOMOGRAPHY"]):
        return "CT"
    if any(x in name for x in ["MR ", "MRI", " MR ", "MAGNETIC RESONANCE"]):
        return "MRI"
    if any(x in name for x in ["PORTABLE", "PORT.", "PORT ", "PA & LAT",
                                "AP & LAT", "AP,LAT", "PRE-OP", "SUPINE",
                                "CHEST (", "RADIOGRAPH", "X-RAY"]):
        return "X-ray"
    if any(x in name for x in [" US", "US.", "ULTRASOUND", "VEINS",
                                "RENAL U", "PELVIS U", "ECHO"]):
        return "Ultrasound"
    if any(x in name for x in ["FLUORO", "LINE PLACEMENT", "GUIDANCE", "PLCT"]):
        return "Fluoroscopy"
    if any(x in name for x in ["NUCLEAR", "SCAN", "PET"]):
        return "Nuclear"
    return "Other"

def assign_body_region(name):
    if is_cpt_code(name):
        return cpt_lookup.get(name.strip(), ("Other", "Other"))[1]
    if any(x in name for x in ["CHEST", "THORAC", "PULM"]):
        return "Chest"
    if any(x in name for x in ["HEAD", "BRAIN", "NEURO", "C-SPINE",
                                "NECK", "CRANIAL", "CEREBR"]):
        return "Head/Neuro"
    if any(x in name for x in ["ABD", "ABDOMEN", "LIVER", "GALLBLADDER",
                                "RENAL", "KIDNEY", "PANCREA", "BILIARY"]):
        return "Abdomen"
    if any(x in name for x in ["PELVIS", "PELVIC", "TRANSVAGINAL", "OB ",
                                "OBSTETRIC", "GYNECOL", "UTERUS", "OVARY"]):
        return "Pelvis/GYN"
    if any(x in name for x in ["SPINE", "L-SPINE", "T-SPINE",
                                "LUMBAR", "THORACIC SPINE"]):
        return "Spine"
    if any(x in name for x in ["CARDIAC", "CORONARY", "HEART", "ECHO"]):
        return "Cardiac"
    if any(x in name for x in ["VASCULAR", "AORTA", "VEINS",
                                "ARTERIO", "CTA"]):
        return "Vascular"
    if any(x in name for x in ["FOOT", "ANKLE", "KNEE", "HIP", "SHOULDER",
                                "ELBOW", "WRIST", "HAND", "FINGER",
                                "LOWER EXT", "UPPER EXT", "FEMUR", "TIBIA"]):
        return "Extremity"
    if any(x in name for x in ["BREAST", "MAMMOGRAM"]):
        return "Breast"
    return "Other"

df_rad["modality"]    = df_rad["exam_name"].apply(assign_modality)
df_rad["body_region"] = df_rad["exam_name"].apply(assign_body_region)

# Audit how many exam_name values still fall through to 'Other' after CPT lookup
other_modality = df_rad[df_rad["modality"] == "Other"]["exam_name"].value_counts()
print(f"\nUnresolved modality (Other): {len(other_modality)} unique values")
print(other_modality.head(20))   # review and extend cpt_lookup or string rules as needed


Unresolved modality (Other): 542 unique values
exam_name
PELVIS, NON-OBSTETRIC                      32581
DIG SCREENING BILAT                        13784
ANKLE (AP, MORTISE & LAT) RIGHT            13010
US GUID FOR VAS. ACCESS                    12376
ANKLE (AP, MORTISE & LAT) LEFT             12278
DIG DIAGNOSTIC MAMMO BILATERAL             11384
WRIST(3 + VIEWS) LEFT                      11370
PELVIS (AP ONLY)                           11160
KNEE (AP, LAT & OBLIQUE) RIGHT             10742
WRIST(3 + VIEWS) RIGHT                     10364
THYROID U.S.                               10249
KNEE (AP, LAT & OBLIQUE) LEFT              10100
CAROTID SERIES COMPLETE                     9612
MRA BRAIN W/O CONTRAST                      9601
ABDOMEN U.S. (COMPLETE STUDY)               9497
HIP UNILAT MIN 2 VIEWS RIGHT                9412
DUPLEX DOPP ABD/PEL                         9216
HIP UNILAT MIN 2 VIEWS LEFT                 9034
C-SPINE NON-TRAUMA 2-3 VIEWS                8880
PARACENTESI

In [27]:
# Flag high-acuity exams
# Covers all three formats: CPT codes, exam codes, and literal names
high_acuity_cpt = {
    "71250", "71260", "71270", "71275",   # CT/CTA chest
    "74177", "74178",                     # CT abdomen/pelvis
    "70450", "70460", "70470",            # CT head
    "70496", "70498",                     # CTA head/neck
    "70551", "70552", "70553",            # MRI brain
    "77001", "77002",                     # line placement
}

high_acuity_strings = [
    "CT HEAD", "CT CHEST", "CT ABD", "CT PELVIS",
    "CTA CHEST", "CTA HEAD", "CTA NECK",
    "MR HEAD", "MRI BRAIN",
    "CHEST (PORTABLE AP)", "CHEST PORT",
    "PORTABLE CHEST", "PORTACATH",
    "COMPUTED TOMOGRAPHY HEAD",
    "COMPUTED TOMOGRAPHY CHEST",
    "FLUORO GUID", "LINE PLACEMENT",
]

def is_high_acuity(name):
    if is_cpt_code(name):
        return int(name.strip() in high_acuity_cpt)
    return int(any(h in name for h in high_acuity_strings))

df_rad["is_high_acuity_exam"] = df_rad["exam_name"].apply(is_high_acuity)

In [28]:
# Aggregate to one row per patient
agg_rad = df_rad.groupby("subject_id").agg(

    # volume
    n_rad_notes          = ("note_id",            "nunique"),
    n_rad_exams          = ("exam_name",           "count"),
    n_high_acuity_exams  = ("is_high_acuity_exam", "sum"),
    has_high_acuity_exam = ("is_high_acuity_exam", "max"),

    # modality flags
    had_ct               = ("modality", lambda x: int("CT"          in x.values)),
    had_cta              = ("modality", lambda x: int("CTA"         in x.values)),
    had_mri              = ("modality", lambda x: int("MRI"         in x.values)),
    had_xray             = ("modality", lambda x: int("X-ray"       in x.values)),
    had_ultrasound       = ("modality", lambda x: int("Ultrasound"  in x.values)),
    had_fluoro           = ("modality", lambda x: int("Fluoroscopy" in x.values)),
    had_nuclear          = ("modality", lambda x: int("Nuclear"     in x.values)),

    # body region flags
    had_chest_imaging    = ("body_region", lambda x: int("Chest"      in x.values)),
    had_neuro_imaging    = ("body_region", lambda x: int("Head/Neuro" in x.values)),
    had_abdomen_imaging  = ("body_region", lambda x: int("Abdomen"    in x.values)),
    had_cardiac_imaging  = ("body_region", lambda x: int("Cardiac"    in x.values)),
    had_vascular_imaging = ("body_region", lambda x: int("Vascular"   in x.values)),
    had_spine_imaging    = ("body_region", lambda x: int("Spine"      in x.values)),
    had_extremity_imaging= ("body_region", lambda x: int("Extremity"  in x.values)),

    # timing — retained for potential interval extension
    first_exam_time      = ("charttime", "min"),
    last_exam_time       = ("charttime", "max"),

).reset_index()

# Exam span: hours between first and last radiology exam
# A wide span suggests prolonged monitoring; a narrow span suggests acute workup
agg_rad["exam_span_hrs"] = (
    (agg_rad["last_exam_time"] - agg_rad["first_exam_time"])
    .dt.total_seconds() / 3600
).round(2)

In [29]:
# Final table
# Drop timestamps from export — retained above only for span calculation
# and future interval work
rad_final = agg_rad.drop(columns=["first_exam_time", "last_exam_time"]).copy()

print(rad_final.shape)
print(rad_final.head())
print(rad_final.describe())

(236847, 20)
   subject_id  n_rad_notes  n_rad_exams  n_high_acuity_exams  \
0    10000032           24           25                    3   
1    10000084            4            4                    3   
2    10000102            1            1                    0   
3    10000108            1            1                    0   
4    10000117           18           20                    1   

   has_high_acuity_exam  had_ct  had_cta  had_mri  had_xray  had_ultrasound  \
0                     1       1        0        0         1               1   
1                     1       1        0        0         1               0   
2                     0       0        0        0         1               0   
3                     0       1        0        0         0               0   
4                     1       1        0        0         1               1   

   had_fluoro  had_nuclear  had_chest_imaging  had_neuro_imaging  \
0           0            0                  1              